[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/20_rl_policy_objectives.ipynb)

# 20. RL policy objectives — token/sequence-level computation

이전 버전의 GRPO는 reward normalization과 scalar ratio만 보여줘 실제 language-model policy objective에서 중요한 **completion token log-probability, old-policy ratio, reference-policy KL, masking**이 빠져 있었다.

이번 버전은 REINFORCE → PPO → DPO → GRPO를 같은 token log-prob 관점에서 연결한다.


In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. REINFORCE on a sequence

autoregressive policy의 trajectory log-probability는 completion token log-prob들의 합이다. terminal reward가 하나라면 같은 return/advantage가 completion 전체의 score-function gradient를 가중한다.


In [ ]:
token_logp = torch.tensor(
    [[-0.5, -0.7, -0.3, -0.4]],
    device=device,
    requires_grad=True,
)
completion_mask = torch.tensor(
    [[1.0, 1.0, 1.0, 0.0]],
    device=device,
)
reward = torch.tensor([1.5], device=device)

sequence_logp = (token_logp * completion_mask).sum(dim=-1)
reinforce_loss = -(reward * sequence_logp).mean()

reinforce_loss.backward()

print("sequence logp:", sequence_logp.detach())
print("token gradients:", token_logp.grad)


## 2. PPO: old-policy importance ratio and clipping

PPO는 data를 생성한 old policy와 현재 policy 사이의 probability ratio를 사용한다. language model에서는 token 단위 ratio를 계산한 뒤 valid completion tokens에서 clipped surrogate를 평균하는 구현이 흔하다.


In [ ]:
old_logp = torch.tensor(
    [
        [-0.8, -0.6, -0.7, -0.2],
        [-0.4, -0.9, -0.5, -0.3],
    ],
    device=device,
)
new_logp = torch.tensor(
    [
        [-0.7, -0.8, -0.6, -0.1],
        [-0.5, -0.7, -0.6, -0.4],
    ],
    device=device,
)
mask = torch.tensor(
    [
        [1.0, 1.0, 1.0, 0.0],
        [1.0, 1.0, 0.0, 0.0],
    ],
    device=device,
)
advantage = torch.tensor(
    [1.0, -0.5],
    device=device,
)[:, None]

ratio = torch.exp(new_logp - old_logp)
clipped_ratio = ratio.clamp(0.8, 1.2)

surrogate = torch.minimum(
    ratio * advantage,
    clipped_ratio * advantage,
)
ppo_loss = -(surrogate * mask).sum() / mask.sum()

print("ratio:\n", ratio)
print("PPO loss:", ppo_loss.item())


## 3. Reference-policy KL estimator

단순히 `mean(policy_logp-ref_logp)`를 KL이라고 부르면 안 된다. sampled actions에서 자주 쓰는 non-negative estimator 중 하나는 `exp(ref_logp-policy_logp) - (ref_logp-policy_logp) - 1`이다. reference policy에서 너무 멀어지는 것을 penalty로 넣을 때 사용한다.


In [ ]:
policy_logp = torch.tensor(
    [[-0.6, -0.4, -0.8]],
    device=device,
)
reference_logp = torch.tensor(
    [[-0.7, -0.5, -0.6]],
    device=device,
)

log_ratio_ref_over_policy = reference_logp - policy_logp
kl_estimator = (
    torch.exp(log_ratio_ref_over_policy)
    - log_ratio_ref_over_policy
    - 1
)

print("sampled-action KL estimator:", kl_estimator)


## 4. DPO: chosen vs rejected relative to a reference model

DPO는 online rollout PPO가 아니라 preference pair에서 policy/reference log-prob margin을 직접 logistic objective로 최적화한다. sequence log-prob를 chosen/rejected 각각 합친 뒤 비교한다.


In [ ]:
pi_chosen = torch.tensor([-3.0, -2.4], device=device)
pi_rejected = torch.tensor([-4.1, -2.9], device=device)
ref_chosen = torch.tensor([-3.3, -2.5], device=device)
ref_rejected = torch.tensor([-3.8, -3.0], device=device)
beta = 0.1

policy_margin = pi_chosen - pi_rejected
reference_margin = ref_chosen - ref_rejected
relative_margin = policy_margin - reference_margin

dpo_loss = -F.logsigmoid(beta * relative_margin).mean()

print("relative preference margin:", relative_margin)
print("DPO loss:", dpo_loss.item())


## 5. GRPO: multiple completions from the same prompt

GRPO의 출발점은 같은 prompt에서 여러 completion을 sampling하고 reward를 **group 내부 상대값**으로 표준화해 advantage를 만드는 것이다. 별도 learned value model을 쓰지 않는 점이 PPO와 큰 차이다.


In [ ]:
# Shape: [num_prompts, group_size]
rewards = torch.tensor(
    [
        [1.2, 0.2, 0.8, 2.0],
        [0.1, 0.4, 0.3, -0.2],
    ],
    device=device,
)

group_mean = rewards.mean(dim=1, keepdim=True)
group_std = rewards.std(
    dim=1,
    keepdim=True,
    unbiased=False,
)
group_advantage = (
    rewards - group_mean
) / (group_std + 1e-6)

print("group advantages:\n", group_advantage)


## 6. GRPO token-level clipped objective + reference KL

각 completion의 scalar group advantage는 그 completion의 valid tokens에 broadcast된다. current/old token log-prob ratio에 PPO-style clipping을 적용하고 reference-policy KL penalty를 함께 더한다.


In [ ]:
num_prompts = 2
group_size = 4
max_tokens = 5

old_logp = -torch.rand(
    num_prompts,
    group_size,
    max_tokens,
    device=device,
)
current_logp = old_logp + 0.15 * torch.randn_like(old_logp)
reference_logp = old_logp + 0.10 * torch.randn_like(old_logp)

completion_mask = torch.tensor(
    [
        [
            [1, 1, 1, 1, 0],
            [1, 1, 1, 0, 0],
            [1, 1, 1, 1, 1],
            [1, 1, 0, 0, 0],
        ],
        [
            [1, 1, 1, 0, 0],
            [1, 1, 1, 1, 0],
            [1, 1, 0, 0, 0],
            [1, 1, 1, 1, 1],
        ],
    ],
    dtype=torch.float32,
    device=device,
)

token_ratio = torch.exp(current_logp - old_logp)
clipped_ratio = token_ratio.clamp(0.8, 1.2)

advantage_per_token = group_advantage[:, :, None]
policy_surrogate = torch.minimum(
    token_ratio * advantage_per_token,
    clipped_ratio * advantage_per_token,
)

ref_over_policy = reference_logp - current_logp
reference_kl = (
    torch.exp(ref_over_policy)
    - ref_over_policy
    - 1
)

kl_beta = 0.04
token_objective = policy_surrogate - kl_beta * reference_kl

grpo_loss = -(
    token_objective * completion_mask
).sum() / completion_mask.sum()

clip_fraction = (
    ((token_ratio < 0.8) | (token_ratio > 1.2)).float()
    * completion_mask
).sum() / completion_mask.sum()

print("GRPO loss:", grpo_loss.item())
print("mean reference KL:", (reference_kl * completion_mask).sum().item() / completion_mask.sum().item())
print("clip fraction:", clip_fraction.item())


## 7. Do not collapse all post-GRPO methods into the same objective

DAPO, Dr.GRPO, GSPO 등은 advantage normalization, clipping, aggregation unit, sequence/token weighting 같은 부분을 서로 다르게 바꾼다. 이 노트북에서는 이름만 나열해서 같은 식인 것처럼 보이게 하지 않고, 원형 GRPO의 실제 tensor path까지만 구현한다.


## References and provenance

**REINFORCE** — Williams. sequence score-function objective를 반영했다.

**PPO** — Schulman et al. old/current policy ratio와 clipped surrogate를 반영했다.

**DPO** — Rafailov et al. chosen/rejected policy margin에서 reference margin을 빼는 pairwise objective를 반영했다.

**GRPO** — DeepSeekMath. 같은 prompt의 group rewards로 relative advantage를 만들고, policy ratio와 KL regularization을 결합하는 구조를 token sequence 형태로 확장해 보여준다. production implementation의 aggregation/estimator 세부사항은 framework마다 조금씩 다를 수 있다.
